# Payment Date Prediction — AI-Enabled B2B Invoice System

**Goal:** Predict the date on which a customer will actually settle an open invoice, so the AR team can prioritise collections on accounts likely to pay late.

**ML Formulation:**  
Regression — predict `payment_delay_days` (actual payment date minus due date).  
Negative = paid early. Positive = paid late.  
For invoices with no `clear_date` yet, we predict the delay and derive a `predicted_payment_date`.

**Pipeline:**
1. Data preparation
2. Exploratory data analysis
3. Feature engineering
4. Model training (Linear Regression vs Random Forest)
5. Evaluation
6. Predict & write results back to MySQL

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Option A – load from CSV export (recommended for offline work)
# Export your DB with: SELECT * FROM winter_internship INTO OUTFILE '/tmp/invoices.csv' ...
# Then place the file at ml/data/invoices.csv

try:
    df = pd.read_csv('../data/invoices.csv', parse_dates=[
        'clear_date', 'posting_date', 'document_create_date',
        'due_in_date', 'baseline_create_date'
    ])
    print('Loaded from CSV')
except FileNotFoundError:
    # Option B – load directly from MySQL
    import mysql.connector, os
    conn = mysql.connector.connect(
        host='localhost', port=3306, database='grey_goose',
        user=os.environ['DB_USER'], password=os.environ['DB_PASSWORD']
    )
    df = pd.read_sql('SELECT * FROM winter_internship', conn,
                     parse_dates=['clear_date', 'posting_date', 'document_create_date',
                                  'due_in_date', 'baseline_create_date'])
    conn.close()
    print('Loaded from MySQL')

print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('=== Null counts ===')
print(df.isnull().sum())
print('\n=== Data types ===')
print(df.dtypes)

In [ ]:
# Proportion of paid vs unpaid invoices
paid   = df['clear_date'].notna().sum()
unpaid = df['clear_date'].isna().sum()
print(f'Paid invoices  : {paid}  ({paid/len(df)*100:.1f}%)')
print(f'Unpaid invoices: {unpaid} ({unpaid/len(df)*100:.1f}%)')

plt.figure(figsize=(5, 3))
plt.bar(['Paid', 'Unpaid'], [paid, unpaid], color=['#34d399', '#f97316'])
plt.title('Invoice Payment Status')
plt.tight_layout()
plt.show()

In [ ]:
# Payment terms distribution
plt.figure(figsize=(8, 3))
df['cust_payment_terms'].value_counts().plot(kind='bar', color='#60a5fa')
plt.title('Customer Payment Terms Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Compute target: how many days late/early was the payment?
paid_df = df[df['clear_date'].notna()].copy()
paid_df['payment_delay_days'] = (
    paid_df['clear_date'] - paid_df['due_in_date']
).dt.days

print('Payment delay stats (days):')
print(paid_df['payment_delay_days'].describe())

plt.figure(figsize=(8, 3))
paid_df['payment_delay_days'].clip(-60, 120).hist(bins=40, color='#60a5fa', edgecolor='#0d1b2a')
plt.axvline(0, color='#f97316', linestyle='--', label='On time')
plt.title('Distribution of Payment Delay (days)')
plt.xlabel('Days (negative = early, positive = late)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def build_features(frame):
    out = frame.copy()

    # Numeric invoice amount
    out['amount'] = pd.to_numeric(out['total_open_amount'], errors='coerce').fillna(0)

    # Days from document creation to due date (payment window length)
    out['days_to_due'] = (
        out['due_in_date'] - out['document_create_date']
    ).dt.days.fillna(0)

    # Days from baseline date to due date (SAP-specific slack)
    out['baseline_slack'] = (
        out['due_in_date'] - out['baseline_create_date']
    ).dt.days.fillna(0)

    # Month of due date (seasonality)
    out['due_month'] = out['due_in_date'].dt.month.fillna(0).astype(int)

    # Encode payment terms as integer (NT30 -> 30, NT60 -> 60, etc.)
    out['terms_days'] = (
        out['cust_payment_terms']
          .str.extract(r'(\d+)')[0]
          .fillna(30)
          .astype(float)
    )

    # Document type encoded
    out['doc_type_enc'] = out['document_type'].astype('category').cat.codes

    return out

features = ['amount', 'days_to_due', 'baseline_slack', 'due_month', 'terms_days', 'doc_type_enc']

paid_df = build_features(paid_df)
paid_df[features].describe()

## 4. Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X = paid_df[features]
y = paid_df['payment_delay_days']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train)} rows | Test: {len(X_test)} rows')

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

for name, model in [('Linear Regression', lr), ('Random Forest', rf)]:
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    print(f'{name:20s}  MAE={mae:.1f} days   RMSE={rmse:.1f} days')

## 5. Feature Importance

In [ ]:
importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(6, 3))
importance.plot(kind='barh', color='#60a5fa')
plt.title('Random Forest — Feature Importance')
plt.tight_layout()
plt.show()

## 6. Predict for Unpaid Invoices & Write Back

In [ ]:
unpaid_df = df[df['clear_date'].isna()].copy()
unpaid_df = build_features(unpaid_df)

delay_pred = rf.predict(unpaid_df[features])
unpaid_df['predicted_payment_date'] = (
    unpaid_df['due_in_date'] + pd.to_timedelta(delay_pred, unit='d')
).dt.strftime('%Y-%m-%d')

print(f'Generated predictions for {len(unpaid_df)} unpaid invoices')
unpaid_df[['sl_no', 'invoice_id', 'due_in_date', 'predicted_payment_date']].head(10)

In [ ]:
# Write predictions back to MySQL
# (Comment this cell out if you only want the CSV output)

import mysql.connector, os

conn = mysql.connector.connect(
    host='localhost', port=3306, database='grey_goose',
    user=os.environ['DB_USER'], password=os.environ['DB_PASSWORD']
)
cursor = conn.cursor()

for _, row in unpaid_df[['sl_no', 'predicted_payment_date']].iterrows():
    cursor.execute(
        'UPDATE winter_internship SET predicted_payment_date = %s WHERE sl_no = %s',
        (row['predicted_payment_date'], int(row['sl_no']))
    )

conn.commit()
cursor.close()
conn.close()

print('Predictions written to winter_internship.predicted_payment_date')

In [ ]:
# Also save as CSV for reference
out_path = '../data/predictions.csv'
unpaid_df[['sl_no', 'invoice_id', 'cust_number', 'due_in_date', 'predicted_payment_date']]\
    .to_csv(out_path, index=False)
print(f'Saved to {out_path}')